# 01 — Profile the Liu 2014 source tables

Controlled first-run profile of Liu et al. (2014), doi:10.1186/1752-0509-8-73. Machinery components (Table S1) and predicted secretory clients (Table S3) are kept separate. Every source row and column is retained.

In [1]:
import json, os, re, sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from atlas.schema import SUBSYSTEM_ORDER

REFERENCE_DIR = Path("../data/reference")
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
WORKBOOK = REFERENCE_DIR / "12918_2013_1339_MOESM2_ESM.xls"
assert WORKBOOK.exists(), f"Missing Liu workbook: {WORKBOOK}"

## Workbook inventory and exact headers

In [2]:
excel = pd.ExcelFile(WORKBOOK)
print(WORKBOOK.name, WORKBOOK.stat().st_size, "bytes")
print("sheets:", excel.sheet_names)

profile = {"source_file": WORKBOOK.name, "format": "xls", "sheets": {}}
for sheet in excel.sheet_names:
    raw = pd.read_excel(WORKBOOK, sheet_name=sheet, header=None)
    profile["sheets"][sheet] = {"physical_rows": len(raw), "physical_columns": raw.shape[1]}
    print(f"{sheet}: {raw.shape[0]} physical rows x {raw.shape[1]} columns")

12918_2013_1339_MOESM2_ESM.xls 3927040 bytes
sheets: ['Table S1', 'Table S2', 'Table S3', 'Table S4', 'Table S5']


Table S1: 371 physical rows x 14 columns


Table S2: 12898 physical rows x 11 columns


Table S3: 2271 physical rows x 9 columns


Table S4: 1020 physical rows x 10 columns


Table S5: 1029 physical rows x 6 columns


In [3]:
# Row 1 is a title; row 2 contains the real headers in Tables S1 and S3.
components = pd.read_excel(WORKBOOK, sheet_name="Table S1", header=1)
clients = pd.read_excel(WORKBOOK, sheet_name="Table S3", header=1)
component_source_columns = components.columns.tolist()
client_source_columns = clients.columns.tolist()
assert len(components) == 369
assert len(clients) == 2269
profile["sheets"]["Table S1"].update({"data_rows": len(components), "columns": component_source_columns})
profile["sheets"]["Table S3"].update({"data_rows": len(clients), "columns": client_source_columns})
print("machinery components:", len(components), component_source_columns)
print("secretory clients:", len(clients), client_source_columns)

machinery components: 369 ['ID', 'S. cerevisiae ortholog', 'Subsystems or function', 'Description', 'CF1.1 vs A1560_logFC', 'A16 vs A1560_logFC', 'CF32 vs A1560_logFC', 'CF1.1vs A1560_adj.P.Val', 'A16 vs A1560_adj.P.Val', 'CF32 vs A1560_adj.P.Val', '1st SOURCE', '2nd SOURCE', '3rd SOURCE', '4th SOURCE']
secretory clients: 2269 ['ID', 'FSD classification', 'CF1.1 vs A1560_logFC', 'A16 vs A1560_logFC', 'CF32 vs A1560_logFC', 'CF1.1vs A1560_adj.P.Val', 'A16 vs A1560_adj.P.Val', 'CF32 vs A1560_adj.P.Val', 'Annotation']


## Identifiers, duplicates, missingness, and paralogues

In [4]:
AO_PATTERN = r"AO090\d{9}"
for label, frame in {"machinery": components, "clients": clients}.items():
    stats = {
        "rows": len(frame),
        "missing_ids": int(frame["ID"].isna().sum()),
        "duplicate_id_rows": int(frame.duplicated("ID", keep=False).sum()),
        "unique_ids": int(frame["ID"].nunique(dropna=True)),
        "ao090_format_matches": int(frame["ID"].astype(str).str.fullmatch(AO_PATTERN).sum()),
    }
    profile[label] = stats
    print(label, stats)

per_yeast = components.groupby("S. cerevisiae ortholog")["ID"].nunique()
profile["machinery"]["yeast_orthologs_with_multiple_ao_genes"] = int((per_yeast > 1).sum())
print("yeast orthologs mapping to >1 A. oryzae ID:", int((per_yeast > 1).sum()))

machinery {'rows': 369, 'missing_ids': 0, 'duplicate_id_rows': 0, 'unique_ids': 369, 'ao090_format_matches': 369}
clients {'rows': 2269, 'missing_ids': 0, 'duplicate_id_rows': 0, 'unique_ids': 2269, 'ao090_format_matches': 2269}
yeast orthologs mapping to >1 A. oryzae ID: 6


## Compare observed subsystem labels with the schema

In [5]:
SUBSYSTEM_NORMALIZATION = {
    "TC": "tc", "Dolichol pathway": "dolichol_pathway",
    "Erglycosylation": "er_glycosylation", "Folding": "folding",
    "GPI biosynthesis": "gpi_biosynthesis", "ERAD": "erad",
    "COPII": "copii", "COPI": "copi",
    "Golgi processing": "golgi_processing", "LDSV": "ldsv",
    "HDSV": "hdsv", "CPY pathway": "cpy_pathway",
    "ALPpathway": "alp_pathway", "SNARE": "snare",
    "Septin": "septin",
    "beta-1,6 glucan biosynthesis": "beta_1_6_glucan_biosynthesis",
    "Translation": "translation",
    "putative mitochondria protein": "putative_mitochondria_protein",
    "mitochondrial m‐AAA protease": "mitochondrial_m_aaa_protease",
}
observed_raw = sorted(components["Subsystems or function"].dropna().unique())
observed_normalized = {SUBSYSTEM_NORMALIZATION[x] for x in observed_raw}
schema_subsystems = set(SUBSYSTEM_ORDER)
profile["subsystems"] = {
    "raw_labels": observed_raw,
    "uncategorized_rows": int(components["Subsystems or function"].isna().sum()),
    "observed_not_in_schema": sorted(observed_normalized - schema_subsystems),
    "schema_not_observed": sorted(schema_subsystems - observed_normalized),
}
print(json.dumps(profile["subsystems"], indent=2, ensure_ascii=False))

{
  "raw_labels": [
    "ALPpathway",
    "COPI",
    "COPII",
    "CPY pathway",
    "Dolichol pathway",
    "ERAD",
    "Erglycosylation",
    "Folding",
    "GPI biosynthesis",
    "Golgi processing",
    "HDSV",
    "LDSV",
    "SNARE",
    "Septin",
    "TC",
    "Translation",
    "beta-1,6 glucan biosynthesis",
    "mitochondrial m‐AAA protease",
    "putative mitochondria protein"
  ],
  "uncategorized_rows": 260,
  "observed_not_in_schema": [],
  "schema_not_observed": []
}


## Verify transcriptomic fields and significance before deriving flags

The paper defines differential expression as adjusted p-value < 0.05. `sig_all_three` therefore requires all three exact adjusted-p columns to be below 0.05. Direction is assigned only when all three corresponding logFC values have the same sign.

In [6]:
LOGFC_COLUMNS = [
    "CF1.1 vs A1560_logFC", "A16 vs A1560_logFC", "CF32 vs A1560_logFC",
]
ADJ_P_COLUMNS = [
    "CF1.1vs A1560_adj.P.Val", "A16 vs A1560_adj.P.Val",
    "CF32 vs A1560_adj.P.Val",
]
SIGNIFICANCE_THRESHOLD = 0.05
for frame in (components, clients):
    assert all(c in frame.columns for c in LOGFC_COLUMNS + ADJ_P_COLUMNS)
    assert all(pd.api.types.is_numeric_dtype(frame[c]) for c in LOGFC_COLUMNS + ADJ_P_COLUMNS)
profile["transcriptomics"] = {
    "logfc_columns": LOGFC_COLUMNS, "adjusted_p_columns": ADJ_P_COLUMNS,
    "criterion": "all three adjusted p-values < 0.05",
    "threshold": SIGNIFICANCE_THRESHOLD,
}
print(json.dumps(profile["transcriptomics"], indent=2))

{
  "logfc_columns": [
    "CF1.1 vs A1560_logFC",
    "A16 vs A1560_logFC",
    "CF32 vs A1560_logFC"
  ],
  "adjusted_p_columns": [
    "CF1.1vs A1560_adj.P.Val",
    "A16 vs A1560_adj.P.Val",
    "CF32 vs A1560_adj.P.Val"
  ],
  "criterion": "all three adjusted p-values < 0.05",
  "threshold": 0.05
}


In [7]:
# Derived only after the exact columns, numeric types, and threshold above are verified.
def add_all_three_flags(frame):
    out = frame.copy()
    out["sig_all_three"] = out[ADJ_P_COLUMNS].lt(SIGNIFICANCE_THRESHOLD).all(axis=1)
    all_up = out[LOGFC_COLUMNS].gt(0).all(axis=1)
    all_down = out[LOGFC_COLUMNS].lt(0).all(axis=1)
    out["direction_all_three"] = pd.Series(None, index=out.index, dtype=object)
    out.loc[out["sig_all_three"] & all_up, "direction_all_three"] = "up"
    out.loc[out["sig_all_three"] & all_down, "direction_all_three"] = "down"
    return out

components = add_all_three_flags(components)
clients = add_all_three_flags(clients)
machinery_counts = components.loc[components.sig_all_three, "direction_all_three"].value_counts().to_dict()
assert int(components.sig_all_three.sum()) == 51
assert machinery_counts == {"up": 48, "down": 3}
profile["derived_counts"] = {
    "machinery_sig_all_three": int(components.sig_all_three.sum()),
    "machinery_direction_all_three": machinery_counts,
    "clients_sig_all_three": int(clients.sig_all_three.sum()),
}
print(profile["derived_counts"])

{'machinery_sig_all_three': 51, 'machinery_direction_all_three': {'up': 48, 'down': 3}, 'clients_sig_all_three': 116}


## Parsing issues, preservation checks, and separate outputs

In [8]:
issues = [
    {"scope": "workbook", "issue": "Tables use a title row above the real header; parsed S1/S3 with header=1"},
    {"scope": "clients", "issue": "p<0.05 in all three workbook columns yields 116 clients; article text reports 111"},
]
for label, frame in (("machinery", components), ("clients", clients)):
    missing_tx = int(frame[LOGFC_COLUMNS + ADJ_P_COLUMNS].isna().any(axis=1).sum())
    if missing_tx:
        issues.append({"scope": label, "issue": f"{missing_tx} rows have at least one missing transcriptomic value"})

assert len(components) == 369 and len(clients) == 2269
assert components.columns[:len(component_source_columns)].tolist() == component_source_columns
assert clients.columns[:len(client_source_columns)].tolist() == client_source_columns
assert set(LOGFC_COLUMNS + ADJ_P_COLUMNS).issubset(components.columns)
assert set(LOGFC_COLUMNS + ADJ_P_COLUMNS).issubset(clients.columns)

components.to_csv(INTERIM_DIR / "liu_components_raw_cleaned.csv", index=False)
clients.to_csv(INTERIM_DIR / "liu_clients_raw_cleaned.csv", index=False)
pd.DataFrame(issues).to_csv(INTERIM_DIR / "parsing_issues.csv", index=False)
profile["parsing_issues"] = issues
with open(INTERIM_DIR / "source_profile.json", "w", encoding="utf-8") as handle:
    json.dump(profile, handle, indent=2, ensure_ascii=False)
print("saved separate machinery and client files; all source rows and columns retained")

saved separate machinery and client files; all source rows and columns retained


## Validated outcomes

- Machinery and clients are saved separately.
- Source row counts and original columns are asserted before export.
- Exact transcriptomic columns and p < 0.05 criterion are recorded before aggregate flags are calculated.
- The machinery result reproduces Liu's 51 total (48 up, 3 down).
- Client-count disagreement with the paper is reported, not silently adjusted.